# AutoGen — Runnable Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anacodicAI-labs/learn-notebooks/blob/main/ai-agents/6-2/autogen.ipynb) &nbsp; [View on GitHub](https://github.com/anacodicAI-labs/learn-notebooks/blob/main/ai-agents/6-2/autogen.ipynb)

Companion to: [AutoGen](https://anacodicai.org/learn/ai-agents/6-2/autogen)


## What you'll build

The **real `autogen-agentchat` library** (the modern AutoGen API — package
name `autogen_agentchat`, not the older `pyautogen`). We build:

1. A single `AssistantAgent` answering a question.
2. A two-agent `RoundRobinGroupChat` — a grader and a reviewer that actually
   converse with each other in turn, AutoGen's core differentiator versus a
   single prompt-in/answer-out chain.

**No API key needed** — `ReplayChatCompletionClient` is a real, built-in
AutoGen model client that plays back a fixed list of responses instead of
calling a network. `pip install autogen-agentchat autogen-ext` if running
locally. AutoGen's agents are `async`, so every call below is awaited.

### Setup

In [1]:
# %pip install -q autogen-agentchat autogen-ext
import asyncio

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import MaxMessageTermination
from autogen_core import CancellationToken
from autogen_ext.models.replay import ReplayChatCompletionClient

print("real autogen-agentchat imported")

real autogen-agentchat imported


## Step 1 — a replay model client

`ReplayChatCompletionClient` takes a plain list of strings and returns them
in order, one per call — it satisfies AutoGen's real model-client interface,
so every agent built on top of it behaves exactly like it would with a real
`OpenAIChatCompletionClient`.

In [2]:
tutor_client = ReplayChatCompletionClient([
    "The capital of France is Paris.",
    "Gradient descent minimizes a loss function by moving parameters "
    "opposite the direction of the gradient.",
])
print(type(tutor_client).__name__, "is a real autogen_ext model client")

ReplayChatCompletionClient is a real autogen_ext model client


## Step 2 — a single AssistantAgent

`on_messages` is the real per-turn entry point AutoGen agents expose. It
takes a list of messages and a `CancellationToken` (also real — used for
timeouts/cancellation in production, unused here) and returns a response
whose `.chat_message` is what the agent said.

In [3]:
tutor = AssistantAgent("tutor", model_client=tutor_client)

async def ask(agent, text):
    response = await agent.on_messages(
        [TextMessage(content=text, source="user")],
        cancellation_token=CancellationToken(),
    )
    return response.chat_message.content

answer_1 = await ask(tutor, "What is the capital of France?")
print("answer 1:", answer_1)

answer_2 = await ask(tutor, "Explain gradient descent in one sentence.")
print("answer 2:", answer_2)

assert "Paris" in answer_1
assert "gradient" in answer_2.lower()
print("\nasserts passed: the agent answered two turns in the order the replay client was given")

answer 1: The capital of France is Paris.
answer 2: Gradient descent minimizes a loss function by moving parameters opposite the direction of the gradient.

asserts passed: the agent answered two turns in the order the replay client was given


## Step 3 — two agents that actually talk to each other

This is what a single LCEL chain or a single CrewAI agent can't do on its
own: a `RoundRobinGroupChat` hands the conversation from one agent to the
next automatically. `MaxMessageTermination(3)` stops after 3 messages —
the task, the grader's turn, and the reviewer's turn.

In [4]:
grader_client = ReplayChatCompletionClient([
    "Draft grade: B. The answer identifies the cause but not the effect.",
])
reviewer_client = ReplayChatCompletionClient([
    "APPROVE: B is consistent with the rubric given the missing effect.",
])

grader = AssistantAgent("grader", model_client=grader_client)
reviewer = AssistantAgent("reviewer", model_client=reviewer_client)

team = RoundRobinGroupChat(
    [grader, reviewer], termination_condition=MaxMessageTermination(3)
)
print("team built with agents:", [a.name for a in [grader, reviewer]])

team built with agents: ['grader', 'reviewer']


## Step 4 — run the team and read the transcript

Watch the `source` on each message: `user` (the task), then `grader`, then
`reviewer` — AutoGen routed the turn automatically, in the order the agents
were listed.

In [5]:
result = await team.run(task="Grade this answer: It rained, so the ground got wet.")

for message in result.messages:
    print(f"{message.source:10}: {message.content}")

assert result.messages[0].source == "user"
assert result.messages[1].source == "grader"
assert result.messages[2].source == "reviewer"
print("\nasserts passed: the group chat routed task -> grader -> reviewer automatically")

user      : Grade this answer: It rained, so the ground got wet.
grader    : Draft grade: B. The answer identifies the cause but not the effect.
reviewer  : APPROVE: B is consistent with the rubric given the missing effect.

asserts passed: the group chat routed task -> grader -> reviewer automatically


## Step 5 — the termination condition actually terminates

**Try it:** change `MaxMessageTermination(3)` to `MaxMessageTermination(2)`
and re-run Step 3+4 — the transcript stops after the grader's turn and the
reviewer never speaks, because the condition is checked after every message,
not just at the end.

In [6]:
print("number of messages in the transcript:", len(result.messages))
assert len(result.messages) == 3
print("assert passed: MaxMessageTermination(3) stopped the conversation at exactly 3 messages")

number of messages in the transcript: 3
assert passed: MaxMessageTermination(3) stopped the conversation at exactly 3 messages


## Try it yourself

- Add a third agent (say, `"instructor"`) to the team and increase
  `MaxMessageTermination` to `4` — watch `RoundRobinGroupChat` add it into
  the rotation with no other code changes.
- Replace `MaxMessageTermination` with a `TextMentionTermination("APPROVE")`
  — the conversation now stops as soon as any agent's message contains that
  word, regardless of turn count.
- Swap `ReplayChatCompletionClient` for
  `autogen_ext.models.openai.OpenAIChatCompletionClient` with a real key —
  the agents, the team, and the termination condition are all unchanged.